# en2zh

In [ ]:
# -*- coding: utf-8 -*-
from openai import OpenAI
from tqdm import tqdm
import random
import time
import base64
import json
import re
import os


def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def get_model_response(system_prompt, user_prompt, model_name="gemini-2.5-flash-preview-05-20", temperature=0.1, seed=42, max_tokens=4096):

    max_retries = 3
    retry_delay = 3  # 初始等待时间（秒）

    for attempt in range(max_retries):
        try:
            # 初始化OpenAI客户端
            client = OpenAI(
                api_key=api_key,
                base_url=api_url,
            )

            # API调用

            response = client.chat.completions.create(
                model="gpt-4o-0806",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {
                        "role": "user",
                        "content": [
                            {
                                'type': 'text',
                                'text': user_prompt
                            },
                        ]
                    },
                ],
                # response_format= {"type": "json_object" },
                # response_format = CalendarEvent,
            )

            # response_json = json.loads(response.choices[0].message.content)
            response_str = response.choices[0].message.content
            return response_str

        except Exception as e:
            if attempt < max_retries:
                wait_time = retry_delay
                time.sleep(wait_time)
            else:
                print(f"TRANSALTE FAILED, Error: {str(e)}")
                raise

def is_chinese(text):
    # 匹配中文字符的正则表达式（包括简体、繁体）
    return bool(re.search(r'[\u4e00-\u9fff]', text))

def is_english(text):
    # 使用正则表达式匹配英文文本
    # 允许英文字母、数字、标点符号和空白字符
    pattern = r'^[a-zA-Z0-9\s.,!?\'\"()\-:;]*$'
    return bool(re.match(pattern, text))

input_file_path = "/home/aiqihang.aqh/Appagent/annotation/result/gpt4o_reason_result.json"        
output_file_path = "/home/aiqihang.aqh/Appagent/annotation/result/gpt4o_reason_result_zh.json"    

with open(input_file_path, "r") as f:
    en_zh_data = json.load(f)

try:
    with open(output_file_path, "r") as f:
        zh_data = json.load(f)
except:
    zh_data = []

output_data = zh_data
done_img_names = [data['image_name'] for data in zh_data]
undone_data = [data for data in en_zh_data if data['image_name'] not in done_img_names]


for data in tqdm(undone_data, total =len(undone_data), desc = "Translating"):

    try:
        try:
            think = data['think']
            answer = data['answer']['content']
        except:
            think, answer = "", ""

        system_prompt = "请把content的内容翻译为中文，如果content本身就是中文，请直接返回原始内容。请直接输出翻译后的中文，不需要带content前缀。"

        if is_english(think):
            user_prompt = f"content: {think}"
            think = get_model_response(system_prompt,user_prompt)
            data['think'] = think

        if is_english(answer):
            user_prompt = f"content: {answer}"
            answer = get_model_response(system_prompt, user_prompt)
            data['answer']['content'] = answer

        output_data.append(data)

    except Exception as e:
        print(f"Error processing data: {str(e)}")
        continue

with open(output_file_path, "w") as f:
    json.dump(output_data, f, indent=4, ensure_ascii=False)
    print(f"Data saved to {output_file_path}")

In [48]:
output_file_path = "/home/aiqihang.aqh/Appagent/annotation/result/gpt4o_reason_result_zh.json"    

with open(input_file_path, "r") as f:
    en_zh_data = json.load(f)

try:
    with open(output_file_path, "r") as f:
        zh_data = json.load(f)
except:
    zh_data = []

zh_data
filtered_zh_data = []
zh_data

for data in zh_data:
    data['instruction'] = data['original_info']['instruction']
    data['type'] = data['answer']['type']
    data['content'] = data['answer']['content']
    data['screenshot'] = data['original_info']['screenshot']
    data['介入原因'] = data['original_info']['介入原因']
    del data['original_info']
    del data['answer']
    filtered_zh_data.append(data)

# filtered_zh_data

with open("/home/aiqihang.aqh/Appagent/annotation/result/gpt4o_reason_result_zh.jsonl", "w", encoding="utf-8") as file:
    for data in filtered_zh_data:
        json.dump(data, file, ensure_ascii=False)
        file.write("\n")

# zh2origin

In [ ]:
# -*- coding: utf-8 -*-
from openai import OpenAI
from tqdm import tqdm
import random
import time
import base64
import json
import re
import os


def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def get_model_response(system_prompt, user_prompt, model_name="gemini-2.5-flash-preview-05-20", temperature=0.1, seed=42, max_tokens=4096):


    max_retries = 3
    retry_delay = 3  # 初始等待时间（秒）

    for attempt in range(max_retries):
        try:
            # 初始化OpenAI客户端
            client = OpenAI(
                api_key=api_key,
                base_url=api_url,
            )

            # API调用

            response = client.chat.completions.create(
                model="gpt-4o-0806",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {
                        "role": "user",
                        "content": [
                            {
                                'type': 'text',
                                'text': user_prompt
                            },
                        ]
                    },
                ],
                # response_format= {"type": "json_object" },
                # response_format = CalendarEvent,
            )

            # response_json = json.loads(response.choices[0].message.content)
            response_str = response.choices[0].message.content
            return response_str

        except Exception as e:
            if attempt < max_retries:
                wait_time = retry_delay
                time.sleep(wait_time)
            else:
                print(f"TRANSALTE FAILED, Error: {str(e)}")
                raise

def read_jsonl(file_path):
    """
    读取 JSONL 文件并返回列表
    
    :param file_path: JSONL 文件路径
    :return: 包含所有 JSON 对象的列表
    """
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON: {e}")
    return data

def is_chinese(text):
    # 匹配中文字符的正则表达式（包括简体、繁体）
    return bool(re.search(r'[\u4e00-\u9fff]', text))

def is_english(text):
    # 使用正则表达式匹配英文文本
    # 允许英文字母、数字、标点符号和空白字符
    pattern = r'^[a-zA-Z0-9\s.,!?\'\"()\-:;]*$'
    return bool(re.match(pattern, text))

In [13]:
input_file_path = '/home/aiqihang.aqh/Appagent/data/【iTAG】交互原因交互内容标注_0530_v1_UTF__20250628102014.jsonl'
output_file_path = '/home/aiqihang.aqh/Appagent/data/【iTAG】交互原因交互内容标注_0530_v2_UTF__20250628102014.jsonl'


all_data = read_jsonl(input_file_path)

for idx,data in tqdm(enumerate(all_data), desc="Processing : "):


    system_prompt = "请把content的内容翻译为英文，如果content本身就是英文，请直接返回原始内容。请直接输出翻译后的英文，不需要带content前缀。"

    user_instruction = data['instruction']
    think = data['交互原因']
    content = data['交互内容']

    # new_data = {
    #     'image': all_data[-1]["image_name"],
    #     'instruction': user_instruction,
    #     'think': think,
    #     'action': {
    #         "action": "call_user",
    #         "content": content
    #     }
    # }

    if is_english(user_instruction):
        user_prompt = f"content: {think}"
        think = get_model_response(system_prompt,user_prompt)
        all_data[idx]['交互原因'] = think
        user_prompt = f"content: {content}"
        content = get_model_response(system_prompt, user_prompt)
        all_data[idx]['交互内容'] = content
    else:
        print("中文指令，无需翻译")

    with open(output_file_path, "a", encoding='utf-8') as file:
        json.dump(all_data[idx], file, ensure_ascii=False)
        file.write("\n")

Processing : : 2it [00:00, 11.88it/s]

中文指令，无需翻译
中文指令，无需翻译
中文指令，无需翻译


Processing : : 5it [00:00, 15.54it/s]

中文指令，无需翻译
中文指令，无需翻译
中文指令，无需翻译
中文指令，无需翻译


Processing : : 8it [00:00, 17.66it/s]

...

Processing : : 897it [16:24,  1.06it/s]

中文指令，无需翻译
中文指令，无需翻译
中文指令，无需翻译
中文指令，无需翻译
中文指令，无需翻译


Processing : : 900it [16:24,  1.66it/s]

中文指令，无需翻译
中文指令，无需翻译
中文指令，无需翻译


Processing : : 903it [16:30,  1.10s/it]


In [6]:
all_data[-1]

{'任务ID': '1928340219616731136',
 '子任务包ID': '1928340377126285312',
 '数据集ID': '1928335789022789632',
 '数据ID': '1928302138305171466',
 'think': '',
 'image_name': 'a873dc73-494e-4762-9a5d-8eb3b568c017.png',
 'instruction': 'I love eating salmon sashimi. Its texture is fresh and tender.',
 'type': 'call_user',
 'content': '',
 'screenshot': 'https://yuqiaochuang-agent.oss-cn-beijing.aliyuncs.com/agent/a873dc73-494e-4762-9a5d-8eb3b568c017.png',
 '介入原因': '["意图确认"]',
 '交互原因': '用户指令为我喜欢吃生鱼片，并未明确操作细节，属于"意图确认"。当前截图在小红书搜索页面，需用户确认意图以便达到用户需求。',
 '交互内容': '请您确认是否搜索有关生鱼片的餐厅还是有关吃生鱼片对身体健康的危害，或者是如何制作生鱼片的食谱？还是其他什么需求？',
 '子任务包状态': '检查中',
 '最终更新时间': '2025-06-10 17:53:45',
 '是否废弃': '否',
 '废弃原因': None,
 '标注环节结果': '[{"userMarkResultId":"1932374455968206848","isNeedVoteJudge":false,"markResultId":"1928340378078392320","markTitle":"交互原因","markResult":"用户指令为我喜欢吃生鱼片，并未明确操作细节，属于\\"意图确认\\"。当前截图在小红书搜索页面，需用户确认意图以便达到用户需求。","questionId":"073c26d5-b688-45f5-a9c3-6664fd40d019","resultType":"INPUT","version":"1749549014091

In [7]:
is_english(all_data[-1]['instruction'])

True